# SC18 - End-to-End Pipeline Results
Runs the **full deployed pipeline** on the test set: YOLO11 detects -> crop from the predicted box ->
ConvNeXt-S (4 heads) classifies -> views aggregated per item -> decision layer. Reports every head
**end-to-end** vs the **oracle-crop** upper bound, the detect rate, and the decision-layer metrics
(coverage / hazard recall / cost). Emits a table (CSV/JSON/LaTeX) + one summary figure.

**Attach:** `thesis-common-10c` + raw images + the ConvNeXt-S weights (`weights_convnext_small...pt`)
+ the trained detector (`yolo11s.pt` from SC8, optional - else it trains a quick one). GPU on.

In [ ]:
import sys, glob, os, json
import numpy as np
# Provenance bootstrap: explicit content identity; no timestamp/size selection.
import hashlib
from pathlib import Path
_PINNED_HELPER_SHA256 = "57b5c16cc9a08c7a3b2afa02c90ece96859b207d4533c78657f862752b68c34b"
ARTIFACT_OVERRIDES = {}  # T.find_one-only paths; generic cached artifact conflicts still require detaching.
_local_input_root = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
_input_roots = [Path('/kaggle/input')] if Path('/kaggle/input').is_dir() else [_local_input_root]
def _content_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
def _input_candidates(pattern):
    hits = set()
    for root in _input_roots:
        for path in root.rglob(pattern):
            if path.is_file() and not {'Archives', '.git', 'node_modules'}.intersection(path.parts):
                hits.add(str(path.resolve()))
    return sorted(hits)
_helpers = _input_candidates('thesis_common.py')
_matching_helpers = [p for p in _helpers if _content_sha256(p) == _PINNED_HELPER_SHA256]
assert _matching_helpers, ('Attach the thesis_common.py supplied in Notebooks. '
                           'No input helper matches SHA256 ' + _PINNED_HELPER_SHA256)
_helper_path = _matching_helpers[0]
sys.path.insert(0, str(Path(_helper_path).parent))
import importlib, thesis_common as T
importlib.reload(T)
assert _content_sha256(T.__file__) == _PINNED_HELPER_SHA256, 'Wrong thesis_common imported.'
def _strict_find_one(filename_glob, required=True):
    if filename_glob in ARTIFACT_OVERRIDES:
        path = Path(ARTIFACT_OVERRIDES[filename_glob])
        assert path.is_file(), 'ARTIFACT_OVERRIDES path missing: ' + str(path)
        return str(path)
    hits = _input_candidates(filename_glob)
    if not hits:
        if required:
            raise FileNotFoundError('Attach the exact producer for ' + filename_glob + '; see the notebook preamble.')
        return None
    if len(hits) > 1:
        digests = {_content_sha256(p) for p in hits}
        if len(digests) != 1:
            raise RuntimeError('Different artifacts share ' + filename_glob + ':\n' + '\n'.join(hits)
                               + '\nDetach stale producers or set ARTIFACT_OVERRIDES explicitly.')
    return hits[0]
T.find_one = _strict_find_one
# Original local find()/find_all() consumers cannot obey T.find_one overrides.
# Reject conflicting standard cached artifacts up front instead of letting those
# consumers silently choose a same-name SC7/SC26 or stale output.
_generic_patterns = ('logits_*.npz', 'metrics_*.json', 'history_*.csv', 'calibration.json', 'config.json')
_by_basename = {}
for _pattern in _generic_patterns:
    for _p in _input_candidates(_pattern):
        _by_basename.setdefault(Path(_p).name, []).append(_p)
for _name, _paths in _by_basename.items():
    if len(_paths) > 1 and len({_content_sha256(p) for p in _paths}) > 1:
        raise RuntimeError('Conflicting input producers for ' + _name + ':\n' + '\n'.join(_paths)
                           + '\nAttach one producer/version per basename for this run.')
print('Pinned helper:', _helper_path, '| SHA256:', _PINNED_HELPER_SHA256)
_cfg = T.find_one('config.json', required=False)
if _cfg: T.load_config(_cfg); print('loaded config:', _cfg)
print('device:', T.DEVICE, '| materials:', len(T.MATERIALS), '| threat levels:', T.NUM_THREAT)

# generic finders across kaggle inputs + local Results
ROOTS = ['/kaggle/input', 'Results', '.']
def find(fn):
    for r in ROOTS:
        h = sorted(glob.glob(f'{r}/**/{fn}', recursive=True), key=len)
        if h: return h[0]
    return None
def find_all(pat):
    seen = {}
    for r in ROOTS:
        for p in sorted(glob.glob(f'{r}/**/{pat}', recursive=True), key=len):
            seen.setdefault(os.path.basename(p), p)
    return list(seen.values())
OUT = '/kaggle/working'
os.makedirs(OUT, exist_ok=True)

In [ ]:
import subprocess, sys, torch, pandas as pd
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics'], check=False)
from ultralytics import YOLO
ROOT = T.find_image_root()

# ---- classifier: ConvNeXt-S (fall back to best available material backbone) ----
BB = 'convnext_small.fb_in22k_ft_in1k'
wpath = T.find_one(f'weights_{BB}.pt', required=False)
if wpath is None:
    _bph = T.find_one('best_per_head.json', required=False)
    candbb = json.load(open(_bph)).get('material') if _bph else None
    for nm in ([candbb] if candbb else []) + T.BACKBONES:
        if nm and T.find_one(f'weights_{nm}.pt', required=False):
            BB = nm; wpath = T.find_one(f'weights_{nm}.pt'); break
assert wpath, 'No classifier weights attached (need weights_<backbone>.pt).'
print('classifier backbone:', BB, '->', wpath)
clf = T.MultiTaskNet(BB).to(T.DEVICE); clf.load_state_dict(torch.load(wpath, map_location=T.DEVICE)); clf.eval()

# ---- detector: use the FINE-TUNED waste detector if attached, else train one ----
# IMPORTANT: a file literally named 'yolo11s.pt' is usually the COCO-pretrained download
# (80 classes), NOT the fine-tuned model. We therefore scan all attached .pt files and
# accept only a checkpoint whose class list is the single 'waste_item' class.
DATA_YAML = T.export_yolo_dataset(ROOT, '/kaggle/working/yolo_ds', include_aug=False)

def is_waste_detector(path):
    try:
        m = YOLO(path)
        names = m.names if isinstance(m.names, dict) else dict(enumerate(m.names))
        return len(names) == 1, m
    except Exception:
        return False, None

det = None
cands  = sorted(glob.glob('/kaggle/input/**/best.pt', recursive=True), key=len)
cands += sorted(glob.glob('/kaggle/input/**/*.pt', recursive=True), key=len)
seen = set()
for p in cands:
    if p in seen or os.path.basename(p).startswith('weights_'): continue
    seen.add(p)
    ok, m = is_waste_detector(p)
    if ok:
        det = m; print('loaded FINE-TUNED detector:', p); break
if det is None:
    print('no fine-tuned detector attached -> training one (yolo11s, 50 epochs)')
    det = YOLO('yolo11s.pt')
    det.train(data=DATA_YAML, epochs=50, imgsz=640, batch=16,
              project=OUT, name='det_e2e', exist_ok=True, verbose=False)
assert len(det.names) == 1, f'detector is not the 1-class waste model: {det.names}'

In [ ]:
# ---- run detect -> crop -> classify over the ORIGINAL test views, collect per-view logits ----
# thesis_common images are NUMPY (H,W,3). Crops reuse T.object_crop (normalized bbox + 20% pad),
# i.e. the SAME preprocessing the classifier was trained with, so the comparison is honest.
import numpy as np, pandas as pd
df = pd.read_csv(os.path.join(ROOT,'test.csv'))
if 'source' in df.columns: df = df[df['source'].fillna('original')=='original'].reset_index(drop=True)

def norm_box_from_xyxy(shape, xyxy):
    H, W = shape[:2]; x1,y1,x2,y2 = xyxy
    return (x1/W, y1/H, (x2-x1)/W, (y2-y1)/H)      # (x,y,w,h) normalized, top-left origin

heads=['material','condition','route','threat']
log_e2e={k:[] for k in heads}; log_ora={k:[] for k in heads}
ys={k:[] for k in heads}; items=[]; detected=0
for _,r in df.iterrows():
    ip = r['image_path'] if os.path.isabs(r['image_path']) else os.path.join(ROOT, r['image_path'])
    if not os.path.exists(ip):
        alt = T.find_one(os.path.basename(r['image_path']), required=False)
        if alt: ip = alt
        else: continue
    im = T.load_rgb(ip)                                   # numpy HWC uint8
    # oracle crop from the GT normalized bbox
    ocrop = T.gray_world(T.object_crop(im, (r['bbox_x'], r['bbox_y'], r['bbox_w'], r['bbox_h']), pad=0.2))
    # detected crop from YOLO's highest-confidence box
    res = det.predict(ip, imgsz=640, conf=0.25, verbose=False)[0]
    if len(res.boxes):
        detected += 1
        b = res.boxes.xyxy[res.boxes.conf.argmax()].tolist()
        dcrop = T.gray_world(T.object_crop(im, norm_box_from_xyxy(im.shape, b), pad=0.2))
    else:
        dcrop = ocrop                                     # detector miss -> still score the item
    with torch.no_grad():
        oo = clf(T.apply_tf(T.eval_tf, ocrop)[None].to(T.DEVICE))
        do = clf(T.apply_tf(T.eval_tf, dcrop)[None].to(T.DEVICE))
    for k in heads:
        log_ora[k].append(oo[k][0].cpu().numpy()); log_e2e[k].append(do[k][0].cpu().numpy())
    ys['material'].append(T.MATERIALS.index(r['material']))
    ys['condition'].append(1 if str(r['condition'])=='Poor' else 0)
    ys['route'].append(T.ROUTES.index(r['route']))
    ys['threat'].append(int(r['threat']))
    items.append(r['item_id'])
items=np.array(items)
for k in heads: log_e2e[k]=np.array(log_e2e[k]); log_ora[k]=np.array(log_ora[k]); ys[k]=np.array(ys[k])
print('views scored:', len(items), '| detect rate:', round(detected/max(len(items),1),4))

In [ ]:
# ---- item-level metrics for oracle vs end-to-end ----
rep_ora = T.item_metrics(log_ora, ys, items)
rep_e2e = T.item_metrics(log_e2e, ys, items)
def flat(rep):
    return {'material_acc':rep['material']['acc'],'material_f1':rep['material']['macro_f1'],
            'condition_auroc':rep['condition']['auroc'],'route_f1':rep['route']['macro_f1'],
            'threat_qwk':rep['threat']['qwk'],'threat_mae':rep['threat']['mae']}
tab = pd.DataFrame({'Oracle crop (GT box)':flat(rep_ora),'End-to-end (detected box)':flat(rep_e2e)}).T
tab['detect_rate']=round(detected/len(items),4)
tab.to_csv(os.path.join(OUT,'end_to_end_results.csv'))
print(tab.round(4).to_string())

In [ ]:
# ---- decision-layer metrics on the end-to-end path ----
agg,uniq,idx = T.aggregate_by_item(log_e2e, items); lab = T.item_labels(ys, items, idx)
mp=T.softmax_np(agg['material']); cp=T.sigmoid_np(agg['condition']); rp=T.softmax_np(agg['route'])
tp=T.corn_class_probs(torch.tensor(agg['threat'])).numpy()
dec = T.decide(mp, cp, tp, route_probs=rp)
rep = T.decision_report_v2(dec, lab['route'], true_threat=lab['threat'])
json.dump({'oracle':flat(rep_ora),'end_to_end':flat(rep_e2e),
           'detect_rate':detected/len(items),'decision':rep},
          open(os.path.join(OUT,'end_to_end_results.json'),'w'), indent=2, default=float)
print('decision layer (end-to-end):',
      {k:rep[k] for k in ['coverage','hazard_recall','total_cost','n'] if k in rep})

In [ ]:
# ---- LaTeX table ----
show=['material_acc','material_f1','condition_auroc','route_f1','threat_qwk','threat_mae']
head=['Mat.\\,Acc','Mat.\\,F1','Cond.\\,AUROC','Route\\,F1','Threat\\,QWK','Threat\\,MAE']
lines=[r'\begin{table}[t]',r'\centering',
       r'\caption{End-to-end pipeline vs the oracle-crop upper bound on the test split. Detect rate '
       + f'{detected/len(items):.3f}'+r'.}',
       r'\label{tab:end_to_end}', r'\begin{tabular}{l'+'c'*len(show)+'}', r'\toprule',
       r'Setting & '+' & '.join(head)+r' \\', r'\midrule']
for nm,rr in [('Oracle crop (GT box)',flat(rep_ora)),('End-to-end (detected box)',flat(rep_e2e))]:
    lines.append(nm+' & '+' & '.join(f'{rr[c]:.3f}' for c in show)+r' \\')
lines+=[r'\bottomrule',r'\end{tabular}',r'\end{table}']
open(os.path.join(OUT,'end_to_end_results.tex'),'w').write('\n'.join(lines))
print('\n'.join(lines))

In [ ]:
# ---- one summary figure: oracle vs end-to-end per head (single, comparison figure) ----
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
keys=['material_acc','material_f1','condition_auroc','route_f1','threat_qwk']
labs=['Mat Acc','Mat F1','Cond AUROC','Route F1','Threat QWK']
o=[flat(rep_ora)[k] for k in keys]; e=[flat(rep_e2e)[k] for k in keys]
x=np.arange(len(keys)); w=0.38
plt.figure(figsize=(7.2,3.8))
plt.bar(x-w/2,o,w,label='Oracle crop',color='#4C78A8')
plt.bar(x+w/2,e,w,label='End-to-end',color='#E45756')
for i,(a,b) in enumerate(zip(o,e)):
    plt.text(i-w/2,a+0.01,f'{a:.2f}',ha='center',fontsize=7); plt.text(i+w/2,b+0.01,f'{b:.2f}',ha='center',fontsize=7)
plt.xticks(x,labs); plt.ylim(0,1.05); plt.ylabel('score'); plt.title('End-to-end vs oracle crop')
plt.legend(); plt.tight_layout(); plt.savefig(os.path.join(OUT,'end_to_end_compare.png'),dpi=200); plt.close()
print('saved end_to_end_compare.png,  end_to_end_results.csv/json/tex')